In [1]:
import numpy as np
import pandas as pd

# Load the datasets created by data_analysis.ipynb.
processed_data = np.load("data/processed_datasets.npz")
X_train = pd.DataFrame(processed_data["X_train"])
X_val = pd.DataFrame(processed_data["X_val"])
X_test = pd.DataFrame(processed_data["X_test"])
Y_train = pd.Series(processed_data["Y_train"], name="churn")
Y_val = pd.Series(processed_data["Y_val"], name="churn")
Y_test = pd.Series(processed_data["Y_test"], name="churn")
processed_data.close()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

X_train: (659028, 44)
X_val: (82379, 44)
X_test: (82379, 44)
Y_train: (659028,)
Y_val: (82379,)
Y_test: (82379,)


In [2]:
print(Y_train.value_counts())

churn
0    593685
1     65343
Name: count, dtype: int64


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [4]:
# Compare default training on the imbalanced data with class-weighted training.
class_counts = Y_train.value_counts()
positive_class_weight = class_counts[0] / class_counts[1]

models = {
    "logistic_regression": {
        "imbalanced": LogisticRegression(max_iter=1000, random_state=42),
        "balanced": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    },
    "random_forest": {
        "imbalanced": RandomForestClassifier(
            n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
        ),
        "balanced": RandomForestClassifier(
            n_estimators=200, max_depth=10, random_state=42, n_jobs=-1,
            class_weight="balanced"
        ),
    },
    "xgboost": {
        "imbalanced": XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, eval_metric="logloss", n_jobs=-1,
            scale_pos_weight=1
        ),
        "balanced": XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            random_state=42, eval_metric="logloss", n_jobs=-1,
            scale_pos_weight=positive_class_weight
        ),
    },
}
print("XGBoost positive class weight:", positive_class_weight)

XGBoost positive class weight: 9.085670997658509


In [5]:
N_FEATURES = 25
assert X_train.shape[1] >= N_FEATURES

In [6]:
def create_pipeline(model):
    return Pipeline([
        ("feature_selection", SelectKBest(score_func=f_classif, k=N_FEATURES)),
        ("model", model),
    ])

In [7]:
pipelines = {
    (model_name, balance): create_pipeline(model)
    for model_name, variants in models.items()
    for balance, model in variants.items()
}
print("\n".join(f"{model_name}_{balance}_top_{N_FEATURES}" for model_name, balance in pipelines))

logistic_regression_imbalanced_top_25
logistic_regression_balanced_top_25
random_forest_imbalanced_top_25
random_forest_balanced_top_25
xgboost_imbalanced_top_25
xgboost_balanced_top_25


In [8]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, roc_auc_score,
)

mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-models-balance-comparison")


def evaluate_model(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
    }


results = []

/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
for (model_name, balance), pipeline in pipelines.items():
    run_name = f"{model_name}_{balance}_top_{N_FEATURES}"
    print("Running:", run_name, flush=True)
    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, Y_train)
        metrics = evaluate_model(pipeline, X_val, Y_val)

        mlflow.log_params({
            "model_type": model_name,
            "balance": balance,
            "n_features": N_FEATURES,
            "decision_threshold": 0.5,
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            pipeline, name="model", serialization_format="cloudpickle"
        )

        results.append({
            "model": model_name,
            "balance": balance,
            "n_features": N_FEATURES,
            **metrics,
        })
        print("Finished:", run_name, "F1:", round(metrics["f1"], 4), flush=True)

Running: logistic_regression_imbalanced_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 17:56:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: logistic_regression_imbalanced_top_25 F1: 0.0058
Running: logistic_regression_balanced_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 17:57:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: logistic_regression_balanced_top_25 F1: 0.2528
Running: random_forest_imbalanced_top_25


...

Finished: random_forest_balanced_top_25 F1: 0.2511
Running: xgboost_imbalanced_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 18:00:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_imbalanced_top_25 F1: 0.014
Running: xgboost_balanced_top_25


/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/28 18:00:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Finished: xgboost_balanced_top_25 F1: 0.2543


In [10]:
results_df = pd.DataFrame(results).sort_values(
    by=["model", "balance"]
).reset_index(drop=True)
results_df.to_csv("data/balance_comparison_results.csv", index=False)
results_df

,model,balance,n_features,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,logistic_regression,balanced,25,0.626482,0.631271,0.157670,0.637243,0.252793,0.682128,0.201936
1,logistic_regression,imbalanced,25,0.900897,0.501334,0.545455,0.002938,0.005845,0.682131,0.202103
2,random_forest,balanced,25,0.613834,0.631278,0.155456,0.653036,0.251130,0.680508,0.201240
3,random_forest,imbalanced,25,0.900921,0.500640,0.687500,0.001347,0.002688,0.679777,0.201956
4,xgboost,balanced,25,0.637808,0.631128,0.159753,0.622796,0.254280,0.680200,0.201894
5,xgboost,imbalanced,25,0.901018,0.503254,0.568627,0.007101,0.014027,0.681608,0.202808
